# DEEP Q-LEARNING for Lunar Landing Environment

The environment is taken from Gymanasium Farama:https://gymnasium.farama.org/environments/box2d/lunar_lander/

Observation space is an `8-dimensional state Vector`

4 Discrete possible Actions:
- 0: do nothing
- 1: fire left orientation engine
- 2: fire main engine
- 3: fire right orientation engine



## Part 0: Installing required packages and Importing Libraries


### Installing gymnasium

In [48]:
!pip install gymnasium
!pip install "gymnasium[atari,accept-rom-licsense]"
!apt-get install -y swig
!pip install gymnasium[box2d]

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
swig is already the newest version (4.0.2-1ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.


### Importing Libraries

In [49]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import autograd as autograd
from torch.autograd import variable
from collections import deque, namedtuple

## Part 1: Building the AI

creating the architecture of the neural network

In [50]:
class Network(nn.Module):

  def __init__(self,state_size,action_size,seed=42):
    super(Network,self).__init__()
    self.seed=torch.manual_seed(seed)

    self.fc1=nn.Linear(state_size,64)
    self.fc2=nn.Linear(64,64)
    self.fc3=nn.Linear(64,action_size)

  def forward(self,state):
    x=F.relu(self.fc1(state))
    x=F.relu(self.fc2(x))
    return self.fc3(x)

## Part 2: Training the AI

Setting up the Environment

In [51]:
import gymnasium as gym
env=gym.make('LunarLander-v3')
state_shape=env.observation_space.shape
state_size=env.observation_space.shape[0]
number_actions=env.action_space.n

print("state_shape: ",state_shape)
print("state_size: ",state_size)
print("Number of actions: ",number_actions)


state_shape:  (8,)
state_size:  8
Number of actions:  4


Initializing the hyperparameters

In [52]:
learning_rate=5e-4
minibatch_size=100
discount_factor=0.99
replay_buffer_size=10000
interpolation_parameter=1e-3

Implementing the experience replay

In [53]:
class ReplayMemory():

  def __init__(self,capacity):
    self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") # check if we have GPU,the use GPU else use CPU
    self.capacity=capacity
    self.memory=[]


  def push(self,event):
    self.memory.append(event)
    if len(self.memory)>self.capacity:
      del self.memory[0]


  def sample(self,batch_size):
    experiences=random.sample(self.memory,k=batch_size)

    states=torch.from_numpy(np.vstack([e[0] for e in experiences if e is not None])).float().to(self.device)
    actions=torch.from_numpy(np.vstack([e[1] for e in experiences if e is not None])).long().to(self.device)
    rewards=torch.from_numpy(np.vstack([e[2] for e in experiences if e is not None])).float().to(self.device)
    next_states=torch.from_numpy(np.vstack([e[3] for e in experiences if e is not None])).float().to(self.device)
    dones=torch.from_numpy(np.vstack([e[4] for e in experiences if e is not None]).astype(np.uint8)).float().to(self.device)


    return states,next_states,actions,rewards,dones

Implementing the DQN Class

In [54]:
class Agent():

  def __init__(self,state_size,action_size):
    self.device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    self.state_size = state_size
    self.action_size = action_size


    self.local_qn = Network(state_size,action_size).to(self.device)
    self.target_qn = Network(state_size,action_size).to(self.device)

    self.optimizer=optim.Adam(self.local_qn.parameters(),lr=learning_rate)
    self.memory = ReplayMemory(replay_buffer_size)
    self.t_step=0


  def step(self,state,action,reward,next_state,done):
    self.memory.push((state,action,reward,next_state,done))
    self.t_step=(self.t_step+1)%4
    if self.t_step==0:
      if len(self.memory.memory) > minibatch_size:
        experiences=self.memory.sample(100)
        self.learn(experiences,discount_factor)

  def act(self,state,epsilon=0.):
    state=torch.from_numpy(state).float().unsqueeze(0).to(self.device)

    self.local_qn.eval()
    with torch.no_grad():
      action_values=self.local_qn(state)

    self.local_qn.train()
    if random.random() > epsilon:
      return np.argmax(action_values.cpu().data.numpy()) #exploit old learnings
    else:
      return random.choice(np.arange(self.action_size)) # explore new possible solutions


  def learn(self,experiences,discount_factor):
    states,next_states,actions,rewards,dones=experiences

    next_q_targets = self.target_qn(next_states).detach().max(1)[0].unsqueeze(1)
    q_targets = rewards + discount_factor * next_q_targets * (1-dones)
    q_expected = self.local_qn(states).gather(1,actions)

    loss = F.mse_loss(q_expected, q_targets)
    self.optimizer.zero_grad()
    loss.backward()
    self.optimizer.step()
    self.soft_update(self.local_qn,self.target_qn,interpolation_parameter)


  def soft_update(self,local_model,target_model,interpolation_parameter):
    for target_param,local_param in zip(target_model.parameters(),local_model.parameters()):
      target_param.data.copy_(interpolation_parameter * local_param.data + (1 - interpolation_parameter) * target_param.data)

Initializing the DQN Agent

In [55]:
agent = Agent(state_size,number_actions)

Training The DQN Agent

In [56]:
# parameters

number_episodes = 2000
maximum_number_tstep_per_episode = 1000

epsilon_start = 1.0
epsilon_decay = 0.995
epsilon_end = 0.01

epsilon = epsilon_start

scores_on_100_epi = deque(maxlen = 100)

In [57]:
# Training Loop

for episode in range(1,number_episodes):
  state,_ = env.reset()
  score = 0

  for t in range(maximum_number_tstep_per_episode):
    action = agent.act(state,epsilon)
    next_state,reward,terminated,truncated,_=env.step(action)
    done = terminated or truncated
    agent.step(state,action,reward,next_state,done)
    state = next_state
    score+=reward

    if done:
      break

  scores_on_100_epi.append(score)
  epsilon = max(epsilon_end, epsilon * epsilon_decay)


# printing scores

  print("\rEpisode {}\tAverage score: {:.2f}".format(episode, np.mean(scores_on_100_epi)),end = "")

  if episode % 100 == 0:
    print("\rEpisode {}\tAverage score: {:.2f}".format(episode, np.mean(scores_on_100_epi)))

  if np.mean(scores_on_100_epi) >= 200.0:
    print("\rEnvironment solved in {:d} episodes!\tAverage Score: {:.2f}".format(episode,np.mean(scores_on_100_epi)))
    torch.save(agent.local_qn.state_dict(), 'checkpoint.oth')
    break

Episode 100	Average score: -150.83
Episode 200	Average score: -129.34
Episode 300	Average score: -39.86
Episode 400	Average score: 60.75
Episode 500	Average score: 66.02
Episode 600	Average score: 160.21
Episode 700	Average score: 105.92
Episode 800	Average score: 87.59
Episode 900	Average score: 154.84
Episode 1000	Average score: 163.61
Environment solved in 1052 episodes!	Average Score: 202.04


## Part 3 : Visualizing the results

In [60]:
import glob
import io
import base64
import imageio
from IPython.display import HTML,display

def show_video_of_model(agent,env_name):
    env = gym.make(env_name,render_mode = 'rgb_array')
    state, _ = env.reset()
    done = False
    frames = []
    while not done :
        frame = env.render()
        frames.append(frame)
        action = agent.act(state)
        state, reward, done, _, _ = env.step(action.item())
    env.close()
    imageio.mimsave('video.mp4', frames, fps=30)

show_video_of_model(agent,'LunarLander-v3')

def show_video():
    mp4list = glob.glob('*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4,'r+b').read()
        encoded = base64.b64encode(video)
        display(HTML(data='''<video alt="test" autoplay
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
                </video>'''.format(encoded.decode('ascii'))))
    else:
        print("could not find video")


show_video()